In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_ENDPOINT = os.getenv("LANGSMITH_ENDPOINT")

In [3]:
import bs4
from langchain_classic import hub
from langsmith import Client
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_openai import OpenAIEmbeddings

/tmp/ipykernel_4637/380011581.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


## Indexing

In [5]:
# load documents
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

In [6]:
# split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

In [7]:
# embed
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(documents=splits,
                                    embedding=embeddings)

In [8]:
retriever = vectorstore.as_retriever()

## Retrieval and Generation 

In [9]:
client = Client(
    api_key=LANGSMITH_API_KEY,
    api_url=LANGSMITH_ENDPOINT
)

In [10]:
# prompt --- won't work in the EU, this one is for the USA only
# prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

In [11]:
import json
from langchain_core.prompts import ChatPromptTemplate

In [12]:
prompt = ChatPromptTemplate.from_template(
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Use three sentences maximum and keep the answer concise.\n"
    "Question: {question} \nContext: {context} \nAnswer:"
)

In [13]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [14]:
# post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [15]:
# chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [16]:
# question
rag_chain.invoke("What is Task Decomposition?")

/home/dmuman/personal/git_repos/ai-engineering-sandbox/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Task decomposition is a planning technique where a complicated, hard task is broken down into smaller, simpler, and more manageable steps. This can be achieved through simple LLM prompting, task-specific instructions, human inputs, or techniques like Chain of Thought. It transforms big tasks into multiple manageable parts to enhance model performance on complex objectives.'

## Indexing under the hood

In [17]:
# documents
question = "What kind of pets do I like?"
document = "My favourite pet is a cat."

In [5]:
import tiktoken

In [19]:
# count tokens considering ~4 char / token
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [20]:
num_tokens_from_string(question, "cl100k_base")

8

In [21]:
# text embedding models
embd = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

3072

### Cosine similarity

In [4]:
import numpy as np

In [23]:
def cosine_similarity(vec1, vec2):
    norm_vec1 = vec1 / np.linalg.norm(vec1)
    norm_vec2 = vec2 / np.linalg.norm(vec2)

    return norm_vec1@norm_vec2

In [24]:
similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.8018380494024135


### Document loaders

In [6]:
### indexing ###
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

`RecursiceCharacterTextSplitter` splits the simple text into paragraphs first, then sentences, then words. It tries to split on them in order until the chunks are small enough. It's trying to keep all paragraphs (and then sentences, and then words) together as long as possible, as those would generally seem to be the strongest semantically related pieces of text.

In [7]:
# split
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

# make splits
splits = text_splitter.split_documents(blog_docs)

In [9]:
# index
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2", api_key=GEMINI_API_KEY)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever()

## Retrieval under the hood

In [29]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # number of documents to search

In [43]:
docs = retriever._get_relevant_documents("What is Task Decomposition?", run_manager="")
print(docs[0])
print(len(docs))

page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner which is common in certain robotic setups but not in many other domains.
Self-Reflection

## Generation under the hood

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate 

In [11]:
# prompt
template = """Answer the question based on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [12]:
# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    api_key=GEMINI_API_KEY
)

In [14]:
# LangChain expression language (LCEL)
# lets easily compose prompts, LLMs, parsers, retrievers and other things
# in the example below I'm connecting a prompt to an llm
chain = prompt | llm

In [16]:
# invoking the chain with a dict
chain.invoke(
    {
        "context":blog_docs,
        "question":"What is Task Decomposition?"
    }
)

/home/dmuman/personal/git_repos/ai-engineering-sandbox/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


AIMessage(content=[{'type': 'text', 'text': 'Based on the provided document, task decomposition refers to the process where an agent breaks down large, complex tasks into smaller, more manageable subgoals or steps, enabling efficient handling of the task. \n\nThis can be accomplished in a few different ways:\n1. **By the LLM itself**, using simple prompting (e.g., "Steps for XYZ." or "What are the subgoals for achieving XYZ?").\n2. **By using task-specific instructions** (e.g., "Write a story outline." for writing a novel).\n3. **With human inputs**. \n\nAdditionally, techniques like **Chain of Thought (CoT)** instruct models to "think step by step" to decompose hard tasks, while **Tree of Thoughts** extends this by exploring multiple reasoning possibilities at each step to create a tree structure.', 'extras': {'signature': 'EnsKeQFpFH0T3IwZuvqWMNIYlccsmtkSGw+skDHd7fuYcLchef/4xlkYHgwaHMtHRvqZ7M4BzDCxgdQ/lAJaT5b771Em6qy+uxSG9vgRcXVqyWly+cqL+xUwWWQEH1KCFYTRJUHqETYjmmw4Ese2cu3eHEsZAD6aRMt

In [ ]:
# rag chain that is going to retrieve the relevant information automatically
# it automatically triggers the retriever which will return documents
# which get passed into the context
rag_chain = (
    {
        "context": retriever, # invokes retriever.invoke(question)
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

`retriever.invoke(question)` is a public analog of `retriever._get_relevant_documents(question)`

use `format_docs()` to merge the `Document` list into a text, so there's no technical metadata of `Document(...)` objects:

```python
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
```

and then:

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    ...
)
```

In [19]:
rag_chain.invoke("What is Task Decomposition?")

/home/dmuman/personal/git_repos/ai-engineering-sandbox/.venv/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3982: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided context, task decomposition is a process where an agent (or LLM) breaks down large, complicated tasks into smaller, simpler, and more manageable steps or subgoals. This enables efficient handling of complex tasks and allows models to utilize more test-time computation to think step by step. \n\nTask decomposition can be performed in the following ways:\n1. By the LLM using simple prompting (e.g., asking for steps or subgoals).\n2. By using task-specific instructions (e.g., "Write a story outline" for writing a novel).\n3. With human inputs.'